# 03 · Fragmentation EDA — RQ1

**Research question:** How much GPU capacity is lost to resource fragmentation,
and how does fragmentation vary across workload types, priority classes, server
tiers, and time?

This notebook produces the four draft figures for the paper and the Table 1
summary statistics.  It also runs the same analysis on the v2023 validation
trace for cross-version comparison.

**Prerequisites:** `make preprocess` completed for both v2026 and v2023.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path so `src` is importable
_root = Path().resolve()
if _root.name == 'notebooks':
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


In [1]:
import sys
print(sys.executable)
print(sys.version)


c:\Users\monis\Documents\Gpu\.venv\Scripts\python.exe
3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]


In [2]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from src.config import (
    INTERIM_DIR, PROCESSED_DIR, FIG_DIR, TABLE_DIR,
    DUCKDB_MEMORY_LIMIT, DUCKDB_THREADS,
)
from src.viz.style import set_ieee_style
import src.analysis.fragmentation as frag
import src.analysis.wasted_hours as wh
from src.viz.figures import (
    fig_frag_cdf_by_workload, fig_frag_heatmap_day_hour,
    fig_frag_by_priority, fig_frag_by_tier,
)

set_ieee_style()
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute(f"SET memory_limit='{DUCKDB_MEMORY_LIMIT}'")
con.execute(f"SET threads={DUCKDB_THREADS}")
print('Ready')

Ready


## Headline numbers — total waste

Total wasted GPU-hours and energy for all three TDP scenarios.

In [3]:
print('=== Waste Report ===')
for r in wh.waste_report(con):
    print(r)

gpu_hours = wh.total_wasted_gpu_hours(con)
kwh_mixed = wh.wasted_kwh(gpu_hours, tdp_watts=wh.MIXED_TDP_W)
co2_kg    = wh.wasted_co2eq(kwh_mixed) / 1000  # kg

print(f'\nHeadline (mixed fleet, cn-beijing grid):')
print(f'  Wasted GPU-hours : {gpu_hours:,.0f}')
print(f'  Wasted energy    : {kwh_mixed:,.1f} kWh')
print(f'  CO₂eq            : {co2_kg/1000:,.2f} tonnes')

=== Waste Report ===


IOException: IO Error: No files found that match the pattern "C:\Users\monis\Documents\Gpu\gpu-carbon-waste\data\processed\hourly_frag.parquet"

LINE 1: SELECT sum(total_frag_gpu_hours) FROM read_parquet('C:\Users\monis\Documents\Gpu\gpu-carbon-waste...
                                              ^

In [ ]:
print('=== Waste by workload type ===')
display(wh.breakdown_by_workload(con))

print('\n=== Waste by priority ===')
display(wh.breakdown_by_priority(con))

## Load analysis datasets

In [ ]:
print('Loading hourly fragmentation …')
hourly_df = frag.compute_frag_rate(con)
print(f'  {len(hourly_df):,} rows  |  frag_rate range: [{hourly_df["frag_rate"].min():.3f}, {hourly_df["frag_rate"].max():.3f}]')

print('Loading per-pod fragmentation …')
pod_df = frag.per_pod_frag_rate(con)
print(f'  {len(pod_df):,} pods  |  workload types: {pod_df["workload_type"].value_counts().to_dict()}')

## Figure 1 — CDF of per-pod fragmentation rate by workload type

In [ ]:
p = fig_frag_cdf_by_workload(pod_df)
print(f'Saved: {p}')
plt.show()

## Figure 2 — Fragmentation heatmap: day × hour-of-day

Look for:
- **Weekly cycles**: lower fragmentation at weekends (reduced job load)?
- **Diurnal cycles**: higher fragmentation during business hours?
- **Campaign spikes**: high-fragmentation blocks corresponding to large training runs?

In [ ]:
hmap_df = frag.day_hour_heatmap(con)
p = fig_frag_heatmap_day_hour(hmap_df)
print(f'Saved: {p}')
plt.show()

# Identify the 5 highest-fragmentation hours
long = hmap_df.reset_index().melt(id_vars='day', var_name='hour_of_day', value_name='frag_rate')
print('\nTop-5 worst (day, hour) cells:')
display(long.nlargest(5, 'frag_rate')[['day','hour_of_day','frag_rate']])

## Figure 3 — Fragmentation by priority class

In [ ]:
p = fig_frag_by_priority(pod_df)
print(f'Saved: {p}')
plt.show()

print('\nFragmentation stats by priority:')
display(frag.aggregate_by_priority(con))

## Figure 4 — Fragmentation by server tier

In [ ]:
p = fig_frag_by_tier(hourly_df)
print(f'Saved: {p}')
plt.show()

print('\nFragmentation stats by server tier:')
display(frag.aggregate_by_server_tier(con))

## Table 1 draft — workload bucket × frag rate percentiles

This becomes Table 1 in Section 4 of the paper.

In [ ]:
table1 = frag.aggregate_by_workload_bucket(con)
print('Table 1: Fragmentation by workload type')
display(table1)
table1.to_csv(TABLE_DIR / 'table1_frag_by_workload.csv', index=False)
print(f'Saved to {TABLE_DIR}/table1_frag_by_workload.csv')

## Diurnal and longitudinal profiles

Supports the narrative in Section 4.2.

In [ ]:
hod = frag.aggregate_by_hour_of_day(con)
day_df = frag.aggregate_by_day(con)

fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.5))

axes[0].fill_between(hod['hour_of_day'], hod['p50_frag_rate'],
                     alpha=0.3, color='#0072B2', label='p50')
axes[0].fill_between(hod['hour_of_day'], hod['p95_frag_rate'],
                     alpha=0.15, color='#D55E00', label='p95')
axes[0].plot(hod['hour_of_day'], hod['mean_frag_rate'],
             color='#0072B2', lw=1.0, label='mean')
axes[0].set_xlabel('Hour of day')
axes[0].set_ylabel('Frag rate')
axes[0].set_title('Diurnal profile')
axes[0].legend()

axes[1].fill_between(day_df['day'], day_df['p50_frag_rate'],
                     alpha=0.3, color='#0072B2')
axes[1].plot(day_df['day'], day_df['mean_frag_rate'],
             color='#0072B2', lw=0.8)
axes[1].set_xlabel('Trace day')
axes[1].set_ylabel('Frag rate')
axes[1].set_title('Longitudinal drift (185 days)')

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_diurnal_longitudinal.pdf')
plt.show()

## v2023 validation — cross-version comparison

Run the same analysis on the v2023 trace (after `make preprocess` for v2023)
and compare fragmentation distributions to check whether v2026 is meaningfully
different or consistent with the earlier trace.

**Expected outcomes:**
- **Similar shape** → fragmentation is a structural property of the scheduler
  (consistent finding across versions).
- **v2026 higher** → cluster has grown faster than scheduling improvements;
  interesting trend to report in Section 7.
- **v2026 lower** → scheduler improvements in v2026; noteworthy if true.

In [ ]:
from src.config import PROCESSED_DIR, INTERIM_DIR
from pathlib import Path

# Paths for v2023 outputs (produced by running the pipeline with --trace v2023)
frag_v2023 = PROCESSED_DIR.parent / 'processed_v2023' / 'hourly_frag.parquet'
pod_v2023  = INTERIM_DIR.parent / 'interim_v2023' / 'pod_hourly_canonical.parquet'

if not frag_v2023.exists():
    print('v2023 processed data not found.')
    print('Run: python -m src.preprocess.pipeline --trace v2023')
    print('(or place v2023 raw data in data/raw/v2023/ and re-run make preprocess)')
else:
    hourly_v2023 = frag.compute_frag_rate(con, frag_src=frag_v2023)
    
    fig, ax = plt.subplots(figsize=(3.5, 2.5))
    for label, df, color in [
        ('v2026 (155K GPUs)', hourly_df,    '#0072B2'),
        ('v2023 (6.2K GPUs)', hourly_v2023, '#E69F00'),
    ]:
        vals = df['frag_rate'].dropna().sort_values()
        cdf  = np.arange(1, len(vals)+1) / len(vals)
        ax.plot(vals, cdf, label=label, color=color, lw=1.0)
    
    ax.set_xlabel('Hourly frag rate')
    ax.set_ylabel('CDF')
    ax.set_title('v2023 vs v2026 fragmentation (sanity check)')
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'fig_v2023_v2026_comparison.pdf')
    plt.show()
    
    # KS test
    from scipy.stats import ks_2samp
    stat, pval = ks_2samp(
        hourly_df['frag_rate'].dropna(),
        hourly_v2023['frag_rate'].dropna(),
    )
    print(f'KS statistic: {stat:.4f}  p-value: {pval:.4e}')
    if pval < 0.05:
        print('  → Distributions are statistically different (worth reporting in §7).')
    else:
        print('  → Cannot reject that distributions are the same.')